# Qwen3-4B with MLX

This notebook loads the local MLX model at `/Users/aayanarish/models/Qwen3-4B-4bit`.

In [1]:
from pathlib import Path
import re
import sys

MODEL_PATH = Path("/Users/aayanarish/models/Qwen3-4B-4bit")

print(sys.executable)
print(sys.version)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Could not find model folder: {MODEL_PATH}")

try:
    from mlx_lm import load, generate
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook needs a Python kernel with mlx-lm installed. "
        "On this Mac, select the Python 3.10 interpreter if VS Code asks for a kernel."
    ) from exc

/usr/local/bin/python3.10
3.10.6 (v3.10.6:9c7b4bd164, Aug  1 2022, 17:13:48) [Clang 13.0.0 (clang-1300.0.29.30)]


In [2]:
model, tokenizer = load(str(MODEL_PATH))
print(f"Loaded {MODEL_PATH.name}")

Loaded Qwen3-4B-4bit


In [3]:
def ask_qwen(message, *, max_tokens=256, think=False, verbose=False):
    thinking_flag = "/think" if think else "/no_think"
    messages = [{"role": "user", "content": f"{message} {thinking_flag}"}]

    if tokenizer.chat_template is not None:
        try:
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=think,
            )
        except TypeError:
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
    else:
        prompt = messages[0]["content"]

    text = generate(
        model,
        tokenizer,
        prompt=prompt,
        max_tokens=max_tokens,
        verbose=verbose,
    )

    if not think:
        text = re.sub(r"<think>.*?</think>\s*", "", text, flags=re.S)
    return text.strip()


print(ask_qwen("Say hello in one short sentence.", max_tokens=40))

Hello!


In [8]:
answer = ask_qwen("start writing code. No asking questions. Just write some (incoherent) code. No english, your output must only be python code", max_tokens=120)
print(answer)

```python
def f(a, b, c):
    d = a + b * c
    e = d % 3
    if e == 0:
        return "zero"
    elif e == 1:
        return "one"
    else:
        return "two"

class x:
    def __init__(self, y):
        self.z = y
        self.v = [1, 2, 3]
        self.w = self.v[1]

def main():
    g = [1, 2, 3]
    h = g[


If you see a `No Metal device available` error, run the notebook in your normal VS Code/Jupyter app rather than this sandboxed Codex environment.